# Segment 4: Tools

## Claude API for Python Developers

In this segment, we'll cover:
- Tool Overview and Use Cases
- Structured Outputs with Tools
- Choosing the Right Tool
- Building Complete Workflows


## Setup and Imports


In [ ]:
import anthropic
import json
from IPython.display import display, Markdown

# Initialize the client
client = anthropic.Anthropic()

print("Client initialized successfully!")


---
## 1. Tool Overview

### What are Tools?

Tools (also called "function calling") allow Claude to:
- Request execution of specific functions
- Receive real-world data to inform responses
- Produce structured, parseable outputs

### How Tools Work:

1. You define available tools with JSON schemas
2. Claude decides when and how to use them
3. You execute the tool and return results
4. Claude incorporates results into its response


In [ ]:
# Define a simple tool - get current weather
weather_tool = {
    "name": "get_weather",
    "description": "Get the current weather for a specific location. Use this when the user asks about weather conditions.",
    "input_schema": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "The city and country, e.g., 'London, UK' or 'New York, US'"
            },
            "unit": {
                "type": "string",
                "enum": ["celsius", "fahrenheit"],
                "description": "Temperature unit preference"
            }
        },
        "required": ["location"]
    }
}

print("Weather tool definition:")
print(json.dumps(weather_tool, indent=2))


In [ ]:
# Make a request with the tool
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    tools=[weather_tool],
    messages=[
        {"role": "user", "content": "What's the weather like in Tokyo?"}
    ]
)

print("Response stop_reason:", response.stop_reason)
print("\nContent blocks:")
for block in response.content:
    print(f"  Type: {block.type}")
    if block.type == "tool_use":
        print(f"  Tool: {block.name}")
        print(f"  Input: {json.dumps(block.input, indent=2)}")
        print(f"  Tool Use ID: {block.id}")


In [ ]:
# Simulate executing the tool and returning results
def get_weather(location, unit="celsius"):
    """Simulated weather function."""
    # In real app, this would call a weather API
    weather_data = {
        "location": location,
        "temperature": 22 if unit == "celsius" else 72,
        "unit": unit,
        "condition": "Partly cloudy",
        "humidity": 65
    }
    return weather_data

# Extract tool use from response
tool_use_block = next(block for block in response.content if block.type == "tool_use")

# Execute the tool
tool_result = get_weather(**tool_use_block.input)
print("Tool execution result:")
print(json.dumps(tool_result, indent=2))


In [ ]:
# Send tool result back to Claude for final response
final_response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    tools=[weather_tool],
    messages=[
        {"role": "user", "content": "What's the weather like in Tokyo?"},
        {"role": "assistant", "content": response.content},
        {
            "role": "user",
            "content": [
                {
                    "type": "tool_result",
                    "tool_use_id": tool_use_block.id,
                    "content": json.dumps(tool_result)
                }
            ]
        }
    ]
)

print("Final response:")
print(final_response.content[0].text)


---
## 2. Structured Outputs

Tools are powerful for forcing Claude to output data in specific formats.


In [ ]:
# Define a tool for structured entity extraction
entity_extraction_tool = {
    "name": "extract_entities",
    "description": "Extract structured information from text including people, organizations, locations, and dates.",
    "input_schema": {
        "type": "object",
        "properties": {
            "people": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {"type": "string"},
                        "role": {"type": "string"}
                    },
                    "required": ["name"]
                },
                "description": "People mentioned in the text"
            },
            "organizations": {
                "type": "array",
                "items": {"type": "string"},
                "description": "Organizations mentioned"
            },
            "locations": {
                "type": "array",
                "items": {"type": "string"},
                "description": "Geographic locations mentioned"
            },
            "dates": {
                "type": "array",
                "items": {"type": "string"},
                "description": "Dates or time periods mentioned"
            }
        },
        "required": ["people", "organizations", "locations", "dates"]
    }
}

# Test with a sample text
text = """
Apple CEO Tim Cook announced at the WWDC conference in San Francisco on June 5, 2024 
that the company would be partnering with OpenAI. The collaboration was praised by 
Microsoft's Satya Nadella, who attended the event alongside Google's Sundar Pichai.
"""

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    tools=[entity_extraction_tool],
    tool_choice={"type": "tool", "name": "extract_entities"},  # Force tool use
    messages=[
        {"role": "user", "content": f"Extract all entities from this text:\n\n{text}"}
    ]
)

# Parse the structured output
tool_use = next(block for block in response.content if block.type == "tool_use")
entities = tool_use.input

print("📊 Extracted Entities:")
print(f"\n👥 People:")
for person in entities["people"]:
    role = f" ({person.get('role', 'N/A')})" if person.get('role') else ""
    print(f"   • {person['name']}{role}")

print(f"\n🏢 Organizations:")
for org in entities["organizations"]:
    print(f"   • {org}")

print(f"\n📍 Locations:")
for loc in entities["locations"]:
    print(f"   • {loc}")

print(f"\n📅 Dates:")
for date in entities["dates"]:
    print(f"   • {date}")


In [ ]:
# Structured output for sentiment analysis
sentiment_tool = {
    "name": "analyze_sentiment",
    "description": "Analyze the sentiment of a text and return structured results.",
    "input_schema": {
        "type": "object",
        "properties": {
            "overall_sentiment": {
                "type": "string",
                "enum": ["very_positive", "positive", "neutral", "negative", "very_negative"],
                "description": "The overall sentiment of the text"
            },
            "confidence": {
                "type": "number",
                "minimum": 0,
                "maximum": 1,
                "description": "Confidence score between 0 and 1"
            },
            "key_phrases": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "phrase": {"type": "string"},
                        "sentiment": {"type": "string", "enum": ["positive", "negative", "neutral"]}
                    },
                    "required": ["phrase", "sentiment"]
                },
                "description": "Key phrases that influenced the sentiment"
            },
            "summary": {
                "type": "string",
                "description": "Brief explanation of the sentiment analysis"
            }
        },
        "required": ["overall_sentiment", "confidence", "key_phrases", "summary"]
    }
}

# Analyze customer reviews
reviews = [
    "The product quality is amazing! Fast shipping and great customer service. Highly recommend!",
    "It's okay, nothing special. Does what it's supposed to do but overpriced.",
    "Terrible experience. Product arrived damaged and support was unhelpful. Never again!"
]

print("📝 Sentiment Analysis Results:\n")

for review in reviews:
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=500,
        tools=[sentiment_tool],
        tool_choice={"type": "tool", "name": "analyze_sentiment"},
        messages=[
            {"role": "user", "content": f"Analyze the sentiment of this review:\n\n{review}"}
        ]
    )
    
    tool_use = next(block for block in response.content if block.type == "tool_use")
    result = tool_use.input
    
    # Map sentiment to emoji
    sentiment_emoji = {
        "very_positive": "🌟", "positive": "😊", 
        "neutral": "😐", "negative": "😞", "very_negative": "😡"
    }
    
    print(f"Review: \"{review[:50]}...\"")
    print(f"  {sentiment_emoji.get(result['overall_sentiment'], '❓')} {result['overall_sentiment'].replace('_', ' ').title()}")
    print(f"  Confidence: {result['confidence']:.0%}")
    print(f"  Summary: {result['summary']}")
    print()


---
## 3. Choosing the Right Tool

When designing tools, consider these factors:
- **Specificity**: More specific tools are easier for Claude to use correctly
- **Descriptions**: Clear descriptions help Claude know when to use each tool
- **Schema design**: Use enums, required fields, and constraints appropriately


In [ ]:
# Multiple tools - Claude chooses the right one
tools = [
    {
        "name": "calculate",
        "description": "Perform mathematical calculations. Use for arithmetic, percentages, etc.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "The mathematical expression to evaluate, e.g., '(10 + 5) * 2'"
                }
            },
            "required": ["expression"]
        }
    },
    {
        "name": "convert_currency",
        "description": "Convert between currencies using current exchange rates.",
        "input_schema": {
            "type": "object",
            "properties": {
                "amount": {"type": "number", "description": "Amount to convert"},
                "from_currency": {"type": "string", "description": "Source currency code (USD, EUR, etc.)"},
                "to_currency": {"type": "string", "description": "Target currency code"}
            },
            "required": ["amount", "from_currency", "to_currency"]
        }
    },
    {
        "name": "get_stock_price",
        "description": "Get the current stock price for a given ticker symbol.",
        "input_schema": {
            "type": "object",
            "properties": {
                "symbol": {"type": "string", "description": "Stock ticker symbol (e.g., AAPL, GOOGL)"}
            },
            "required": ["symbol"]
        }
    }
]

# Test with different queries - Claude picks the right tool
test_queries = [
    "What is 15% of 200?",
    "Convert 100 USD to EUR",
    "What's the current price of Apple stock?"
]

for query in test_queries:
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=500,
        tools=tools,
        messages=[{"role": "user", "content": query}]
    )
    
    print(f"Query: {query}")
    for block in response.content:
        if block.type == "tool_use":
            print(f"  → Claude chose: {block.name}")
            print(f"  → Parameters: {json.dumps(block.input)}")
    print()


In [ ]:
# Tool choice options
print("Tool Choice Options:")
print("=" * 50)

# Option 1: auto (default) - Claude decides whether to use a tool
print("\n1. auto (default) - Claude decides")
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=200,
    tools=tools,
    tool_choice={"type": "auto"},  # Default behavior
    messages=[{"role": "user", "content": "Hello, how are you?"}]
)
print(f"   Stop reason: {response.stop_reason}")
print(f"   Response: {response.content[0].text if response.content[0].type == 'text' else 'Tool use'}")

# Option 2: any - Force Claude to use some tool
print("\n2. any - Must use some tool")
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=200,
    tools=tools,
    tool_choice={"type": "any"},
    messages=[{"role": "user", "content": "I want to know something about money"}]
)
tool_block = next((b for b in response.content if b.type == "tool_use"), None)
if tool_block:
    print(f"   Tool used: {tool_block.name}")

# Option 3: tool - Force specific tool
print("\n3. tool - Force specific tool")
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=200,
    tools=tools,
    tool_choice={"type": "tool", "name": "calculate"},
    messages=[{"role": "user", "content": "What's the weather like?"}]  # Irrelevant query
)
tool_block = next((b for b in response.content if b.type == "tool_use"), None)
if tool_block:
    print(f"   Forced tool: {tool_block.name}")
    print(f"   Input: {tool_block.input}")


---
## 4. Building Complete Workflows

Let's build a complete agent that can use multiple tools to accomplish tasks.


In [ ]:
# Define a set of tools for a simple assistant
assistant_tools = [
    {
        "name": "search_database",
        "description": "Search the product database for items matching a query.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search query"},
                "category": {
                    "type": "string",
                    "enum": ["electronics", "clothing", "books", "all"],
                    "description": "Product category to search"
                },
                "max_results": {"type": "integer", "default": 5}
            },
            "required": ["query"]
        }
    },
    {
        "name": "get_product_details",
        "description": "Get detailed information about a specific product.",
        "input_schema": {
            "type": "object",
            "properties": {
                "product_id": {"type": "string", "description": "The product ID"}
            },
            "required": ["product_id"]
        }
    },
    {
        "name": "add_to_cart",
        "description": "Add a product to the shopping cart.",
        "input_schema": {
            "type": "object",
            "properties": {
                "product_id": {"type": "string"},
                "quantity": {"type": "integer", "default": 1}
            },
            "required": ["product_id"]
        }
    },
    {
        "name": "check_inventory",
        "description": "Check if a product is in stock.",
        "input_schema": {
            "type": "object",
            "properties": {
                "product_id": {"type": "string"}
            },
            "required": ["product_id"]
        }
    }
]

# Simulated tool implementations
def execute_tool(name, inputs):
    """Execute a tool and return simulated results."""
    if name == "search_database":
        return {
            "results": [
                {"id": "P001", "name": "Wireless Headphones", "price": 79.99},
                {"id": "P002", "name": "Bluetooth Speaker", "price": 49.99},
                {"id": "P003", "name": "USB-C Hub", "price": 34.99}
            ]
        }
    elif name == "get_product_details":
        return {
            "id": inputs["product_id"],
            "name": "Wireless Headphones",
            "price": 79.99,
            "description": "High-quality wireless headphones with noise cancellation",
            "rating": 4.5,
            "reviews": 1250
        }
    elif name == "add_to_cart":
        return {"success": True, "message": f"Added {inputs.get('quantity', 1)} item(s) to cart"}
    elif name == "check_inventory":
        return {"in_stock": True, "quantity_available": 50}
    return {"error": "Unknown tool"}

print("🛠️ Assistant tools defined:")
for tool in assistant_tools:
    print(f"  • {tool['name']}: {tool['description'][:50]}...")


In [ ]:
def run_agent(user_message, tools, max_iterations=5):
    """
    Run a simple agent loop that handles tool calls.
    """
    messages = [{"role": "user", "content": user_message}]
    
    print(f"👤 User: {user_message}")
    print("-" * 50)
    
    for iteration in range(max_iterations):
        # Get response from Claude
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1024,
            tools=tools,
            messages=messages
        )
        
        # Check if Claude wants to use tools
        if response.stop_reason == "tool_use":
            # Process each tool use
            tool_results = []
            
            for block in response.content:
                if block.type == "tool_use":
                    print(f"🔧 Tool: {block.name}")
                    print(f"   Input: {json.dumps(block.input)}")
                    
                    # Execute the tool
                    result = execute_tool(block.name, block.input)
                    print(f"   Result: {json.dumps(result)[:100]}...")
                    
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(result)
                    })
            
            # Add assistant response and tool results to messages
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})
            
        else:
            # Claude is done with tools, return final response
            final_text = next(
                (block.text for block in response.content if block.type == "text"),
                "No response"
            )
            print("-" * 50)
            print(f"🤖 Assistant: {final_text}")
            return final_text
    
    return "Max iterations reached"

# Test the agent
print("\n" + "=" * 60)
print("AGENT DEMO")
print("=" * 60 + "\n")

result = run_agent(
    "I'm looking for wireless audio devices. Can you show me what's available?",
    assistant_tools
)


In [ ]:
# Test with a multi-step task
print("\n" + "=" * 60)
print("MULTI-STEP TASK")
print("=" * 60 + "\n")

result = run_agent(
    "Can you find wireless headphones, check if they're in stock, and add 2 to my cart?",
    assistant_tools
)


### Real-World Tool Example: Data Transformation


In [ ]:
# Tool for structured data extraction from unstructured text
data_extraction_tool = {
    "name": "extract_invoice_data",
    "description": "Extract structured data from invoice text.",
    "input_schema": {
        "type": "object",
        "properties": {
            "invoice_number": {"type": "string"},
            "date": {"type": "string", "description": "Invoice date in YYYY-MM-DD format"},
            "vendor": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "address": {"type": "string"}
                },
                "required": ["name"]
            },
            "line_items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "description": {"type": "string"},
                        "quantity": {"type": "integer"},
                        "unit_price": {"type": "number"},
                        "total": {"type": "number"}
                    },
                    "required": ["description", "quantity", "unit_price", "total"]
                }
            },
            "subtotal": {"type": "number"},
            "tax": {"type": "number"},
            "total": {"type": "number"}
        },
        "required": ["invoice_number", "date", "vendor", "line_items", "total"]
    }
}

# Sample invoice text
invoice_text = """
INVOICE #2024-1234

From: Tech Supplies Inc.
123 Business Park, Suite 456
San Francisco, CA 94102

Date: December 3, 2024

ITEMS:
- Laptop Stand (x2) @ $45.00 each = $90.00
- USB-C Cable 6ft (x5) @ $12.99 each = $64.95  
- Wireless Mouse (x1) @ $29.99 each = $29.99

Subtotal: $184.94
Tax (8.5%): $15.72
TOTAL DUE: $200.66
"""

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    tools=[data_extraction_tool],
    tool_choice={"type": "tool", "name": "extract_invoice_data"},
    messages=[
        {"role": "user", "content": f"Extract the data from this invoice:\n\n{invoice_text}"}
    ]
)

tool_use = next(block for block in response.content if block.type == "tool_use")
invoice_data = tool_use.input

print("📄 Extracted Invoice Data:")
print(json.dumps(invoice_data, indent=2))


---
## Summary

In this segment, we covered:

1. **Tool Overview**: How to define and use tools with JSON schemas
2. **Structured Outputs**: Forcing specific output formats for reliable parsing
3. **Choosing Tools**: Using `tool_choice` to control tool selection
4. **Complete Workflows**: Building agent loops that handle multi-step tasks

### Key Takeaways:
- Tools enable Claude to interact with external systems
- JSON schemas define tool inputs precisely
- Use `tool_choice` to control when/which tools are used
- Agent loops handle the tool-use → result → response cycle
- Structured outputs are reliable for data extraction

### Best Practices:
- Write clear, specific tool descriptions
- Use enums and required fields to constrain inputs
- Handle tool errors gracefully
- Limit agent iterations to prevent infinite loops
- Validate tool outputs before using them

### Next Steps
In Segment 5, we'll explore code generation and explanation:
- Generating code from prompts
- Explaining existing code
- Adding documentation and comments
